In [1]:
# Library Imports.

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import scipy.fftpack as fftpack
import joblib
from keras.layers import Input, Conv2D, Activation, Add, Lambda
import keras.backend as K
from keras.models import Model
import pickle
import keras
from keras.optimizers import Adam
import tensorflow as tf
# import os


# Class/Function Imports
from SVM import compute_dct_features, extract_features, PerturbationDetectorSVM
from PRN import PRN
from PRN_augmented import UNet_PRN

Using TensorFlow backend.


In [ ]:
BATCH_SIZE = 64
LEARNING_RATE = 0.01
MAX_EPOCH = 12

# PRN and Joint Model Training

In [ ]:
# Create & Import models
file_save_path = "results/cifar_res.p"
with open(file_save_path, 'rb') as f:
    honeypot_data = pickle.load(f)

prn = UNet_PRN(input_shape=(32, 32, 3))

model_file = honeypot_data['model_file']
print(f"Loading Honeypot model from: {model_file}")

honeypot_model = keras.models.load_model(model_file, compile=False) 

honeypot_model.trainable = False

joint_inputs = Input(shape=(32, 32, 3))
cleaned_images = prn(joint_inputs)
final_predictions = honeypot_model(cleaned_images)

joint_model = Model(inputs=joint_inputs, outputs=final_predictions)

joint_model.compile(
    optimizer=Adam(lr=0.0001), 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

joint_model.summary()

mixed_train_X = np.load('./data/mixed_train_X.npy') 
clean_train_Y = np.load('./data/mixed_train_Y.npy')

joint_model._make_train_function()
K.get_session().run(tf.compat.v1.global_variables_initializer())
honeypot_model.load_weights(model_file)

joint_model.fit(
    x=mixed_train_X, 
    y=clean_train_Y, 
    batch_size=BATCH_SIZE, 
    epochs=MAX_EPOCH,
)

save_dir = 'models'
os.makedirs(save_dir, exist_ok=True)

prn.save('models/trained_prn_test1.h5')
joint_model.save('models/trained_joint_model_test1.h5')






Loading Honeypot model from: models/cifar_model.h5








Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         (None, 32, 32, 3)         0         
_________________________________________________________________
UNet_PRN (Model)             (None, 32, 32, 3)         472387    
_________________________________________________________________
sequential_1 (Sequential)    (None, 10)                2923050   
Total params: 3,395,437
Trainable params: 472,387
Non-trainable params: 2,923,050
_________________________________________________________________
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Epoch 1/8
50000/50000 [==============================] - 249s 5ms/step - loss: 2.8347 - acc: 0.5668
Epoch 2/8
500

# SVM Training

In [ ]:
mixed_test_X = np.load('./data/mixed_test_X.npy')
clean_test_X = np.load('./data/clean_test_X.npy')
clean_test_Y = np.load('./data/mixed_test_Y.npy')
test_attack_labels = np.load('./data/test_attack_labels.npy')
mixed_train_X = np.load('./data/mixed_train_X.npy')
train_attack_labels = np.load('./data/train_attack_labels.npy')

prn = tf.keras.models.load_model('./models/trained_prn_augmented.h5', compile=False)

clean_idx_train = np.where(train_attack_labels == 0)[0]
adv_idx_train = np.where(train_attack_labels == 1)[0]

clean_train_imgs = mixed_train_X[clean_idx_train]
adv_train_imgs = mixed_train_X[adv_idx_train]

svm_detector = PerturbationDetectorSVM(prn, device=None, keep_coeffs=8) 
svm_detector.fit(clean_train_imgs, adv_train_imgs)

save_dir = 'models'
os.makedirs(save_dir, exist_ok=True)

svm_detector.save('models/svm_detector_test1.joblib')

clean_idx_test = np.where(test_attack_labels == 0)[0]
adv_idx_test = np.where(test_attack_labels == 1)[0]

clean_test_imgs = mixed_test_X[clean_idx_test]
adv_test_imgs = mixed_test_X[adv_idx_test]


svm_metrics = svm_detector.evaluate(clean_test_imgs, adv_test_imgs)

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
Training accuracy: 0.9808
Detector Evaluation
----------------------------------------
False Positive Rate (FPR): 0.0340
True Positive Rate  (TPR): 0.9656
False Negative Rate (FNR): 0.0344
True Negative Rate  (TNR): 0.9660
----------------------------------------
              precision    recall  f1-score   support

       clean       0.97      0.97      0.97      5028
 adversarial       0.97      0.97      0.97      4972

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000

